# Read data from database


In [20]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [2]:
load_dotenv('../.env')
USERNAME = os.getenv('DB_USERNAME')
PASSWORD = os.getenv('PASSWORD')
HOST = os.getenv('HOST')
PORT = os.getenv('PORT')
DATABASE = os.getenv('DATABASE')

In [5]:
engine = create_engine(f"mysql+pymysql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

champion_team_player_df = pd.read_sql('SELECT * FROM championteamplayer', engine)
college_df              = pd.read_sql('SELECT * FROM college', engine)
country_df              = pd.read_sql('SELECT * FROM country', engine)
mvp_award_df            = pd.read_sql('SELECT * FROM mvp_award', engine)
player_df               = pd.read_sql('SELECT * FROM player', engine)
player_college_df       = pd.read_sql('SELECT * FROM playercollege', engine)
player_position_df      = pd.read_sql('SELECT * FROM playerposition', engine)
player_season_stats_df  = pd.read_sql('SELECT * FROM playerseasonstats', engine)
positions_df            = pd.read_sql('SELECT * FROM position', engine)
season_df               = pd.read_sql('SELECT * FROM season', engine)
team_df                 = pd.read_sql('SELECT * FROM team', engine)


# Preprocessing

In [39]:
def check_fk(child_df, child_col, parent_df, parent_col, child_name, parent_name):
    child_keys = set(child_df[child_col].dropna().unique())
    parent_keys = set(parent_df[parent_col].dropna().unique())
    missing = child_keys - parent_keys
    print(f"{child_name}.{child_col} -> {parent_name}.{parent_col}: "
          f"{len(missing)} unmatched value(s)")
    if missing:
        print("  examples:", list(missing)[:10])
        # show the actual offending rows
        bad_rows = child_df[child_df[child_col].isin(missing)]
        print(bad_rows)
    return missing

### champion_team_player_df

In [6]:
champion_team_player_df.head()

,player_id,season_id,team_code,jersey_number,age,experience,games,minutes,win_shares
0,adamsja01,2,MIL,None,None,None,None,None,None
1,antetgi01,2,MIL,None,None,None,None,None,None
2,antetth01,2,MIL,None,None,None,None,None,None
3,augusdj01,2,MIL,None,None,None,None,None,None
4,bantoda01,5,BOS,None,None,None,None,None,None


In [41]:
print(champion_team_player_df.isnull().sum())
print('---')
print(f'Duplicated: {champion_team_player_df.duplicated().sum()}')
print('---')
# ChampionTeamPlayer checks
check_fk(champion_team_player_df, 'player_id', player_df, 'player_id', 'ChampionTeamPlayer', 'Player')
check_fk(champion_team_player_df, 'season_id', season_df, 'season_id', 'ChampionTeamPlayer', 'Season')
check_fk(champion_team_player_df, 'team_code', team_df,   'team_code', 'ChampionTeamPlayer', 'Team')


player_id         0
season_id         0
team_code         0
jersey_number    76
age              76
experience       76
games            76
minutes          76
win_shares       76
dtype: int64
---
Duplicated: 0
---
ChampionTeamPlayer.player_id -> Player.player_id: 0 unmatched value(s)
ChampionTeamPlayer.season_id -> Season.season_id: 0 unmatched value(s)
ChampionTeamPlayer.team_code -> Team.team_code: 0 unmatched value(s)


set()

### college_df

In [7]:
college_df.head()

,college_id,college_name
0,1,1985in College Park
1,2,Acadia University
2,3,Air Force
3,4,Alabama
4,5,Alabama A&M University


In [54]:
print('shape:', college_df.shape)
print('---')
print(college_df.isna().sum())
print('---')
print('duplicate:', college_df.duplicated().sum())

shape: (670, 2)
---
college_id      0
college_name    0
dtype: int64
---
duplicate: 0


### country

In [8]:
country_df.head()

,country_id,country_code
0,1,ag
1,2,ao
2,3,ar
3,4,at
4,5,au


In [55]:
print('shape:', country_df.shape)
print('---')
print(country_df.isna().sum())
print('---')
print('duplicate:', country_df.duplicated().sum())

shape: (93, 2)
---
country_id      0
country_code    0
dtype: int64
---
duplicate: 0


### mvp_award_df

In [56]:
mvp_award_df.head()

,player_id,season_id,mvp_rank,mvp_share
0,antetgi01,1,1.0,0.952
1,antetgi01,2,4.0,0.345
2,antetgi01,3,3.0,0.595
3,antetgi01,4,3.0,0.606
4,antetgi01,5,4.0,0.194


In [49]:
print(f'shape:', mvp_award_df.shape)
print('---')
print(mvp_award_df.isnull().sum())
print('---')
print(f'Duplicated: {mvp_award_df.duplicated().sum()}')
print('---')
# ChampionTeamPlayer checks
check_fk(mvp_award_df, 'player_id', player_df, 'player_id', 'ChampionTeamPlayer', 'Player')
check_fk(mvp_award_df, 'season_id', season_df, 'season_id', 'ChampionTeamPlayer', 'Season')

shape: (61, 4)
---
player_id    0
season_id    0
mvp_rank     8
mvp_share    0
dtype: int64
---
Duplicated: 0
---
ChampionTeamPlayer.player_id -> Player.player_id: 0 unmatched value(s)
ChampionTeamPlayer.season_id -> Season.season_id: 0 unmatched value(s)


set()

In [50]:
print(mvp_award_df.value_counts('mvp_rank'))
mvp_award_df[mvp_award_df['mvp_rank'].isna()]

mvp_rank
1.0     5
4.0     5
3.0     5
5.0     5
8.0     5
9.0     5
6.0     5
7.0     5
2.0     5
11.0    3
10.0    3
12.0    2
Name: count, dtype: int64


,player_id,season_id,mvp_rank,mvp_share
6,brunsja01,4,NaN,0.001
14,derozde01,3,NaN,0.001
20,duranke01,3,NaN,0.001
31,hardeja01,2,NaN,0.001
33,jamesle01,2,NaN,0.001
34,jamesle01,3,NaN,0.001
41,leonaka01,2,NaN,0.001
46,moranja01,4,NaN,0.001


In [53]:
print(mvp_award_df['mvp_rank'].isna().sum(), 'out of', len(mvp_award_df))
print(mvp_award_df[mvp_award_df['mvp_rank'].isnull()].groupby('season_id').size())


8 out of 61
season_id
2    3
3    3
4    2
dtype: int64


### player_df

In [10]:
player_df.head()

,player_id,full_name,birth_date,birth_place,height_cm,weight_kg,shoots,player_url,country_id
0,abdelal01,Alaa Abdelnaby,1968-06-24,"Cairo,Egypteg",208,108.0,Right,https://www.basketball-reference.com/players/a...,27
1,abdulka01,Kareem Abdul-Jabbar,1947-04-16,"New York,New Yorkus",218,102.0,Right,https://www.basketball-reference.com/players/a...,86
2,abdulma01,Walt Hazzard,1942-04-15,"Wilmington,Delawareus",188,83.0,Right,https://www.basketball-reference.com/players/a...,86
3,abdulma02,Mahmoud Abdul-Rauf,1969-03-09,"Gulfport,Mississippius",185,73.0,Right,https://www.basketball-reference.com/players/a...,86
4,abdulta01,Tariq Abdul-Wahad,1974-11-03,"Maisons Alfort,Francefr",198,101.0,Right,https://www.basketball-reference.com/players/a...,30


In [57]:
print('shape:', player_df.shape)
print('---')
print(player_df.isna().sum())
print('---')
print('Duplicated:', player_df.duplicated().sum())

shape: (5416, 9)
---
player_id        0
full_name        0
birth_date     111
birth_place      4
height_cm        0
weight_kg        5
shoots           0
player_url       0
country_id       0
dtype: int64
---
Duplicated: 0


In [73]:
# Extracting players who their 'weight_kg' is missing
missing_weight_kg = player_df[player_df['weight_kg'].isna()]
print(missing_weight_kg[['player_id', 'player_url']])

# I check the actual url for these 5 players. there is nothing wrong with scraping. 
# four of them are dead ;(

player_df = player_df[player_df['weight_kg'].notna()]
print(player_df.shape)

Empty DataFrame
Columns: [player_id, player_url]
Index: []
(5411, 9)


In [76]:
# Extracting players who dosn't have 'birth_place' 
missing_birth_place = player_df[player_df['birth_place'].isna()]
missing_birth_place
# I Check the site, there is a bug in scraping players data 

,player_id,full_name,birth_date,birth_place,height_cm,weight_kg,shoots,player_url,country_id
776,carusal01,Alex Caruso,None,NaN,196,84.0,Right,https://www.basketball-reference.com/players/c...,86
2782,lawalga01,Gani Lawal,None,NaN,206,106.0,Right,https://www.basketball-reference.com/players/l...,86
4479,smithjo03,Josh Smith,None,NaN,206,102.0,Left,https://www.basketball-reference.com/players/s...,86


### player_college_df

In [11]:
player_college_df.head()

,player_id,college_id
0,smithjo03,1
1,heanebr01,2
2,moonema01,3
3,anslemi01,4
4,askinke01,4


In [77]:
print('shape:', player_college_df.shape)
print('---')
print(player_college_df.isna().sum())
print('---')
print('Duplicated:', player_college_df.duplicated().sum())

shape: (5751, 2)
---
player_id     0
college_id    0
dtype: int64
---
Duplicated: 0


### player_position_df

In [12]:
player_position_df.head()

,player_id,position_id
0,abdulka01,1
1,abdulza01,1
2,abdursh01,1
3,achiupr01,1
4,acresma01,1


In [78]:
print('shape:', player_position_df.shape)
print('---')
print(player_position_df.isna().sum())
print('---')
print('Duplicated:', player_position_df.duplicated().sum())

shape: (7156, 2)
---
player_id      0
position_id    0
dtype: int64
---
Duplicated: 0


### player_season_stats_df

In [13]:
player_season_stats_df.head()

,player_id,season_id,team_code,age,position_code,games,games_started,minutes,win_shares,ws_per_48,bpm,vorp
0,achiupr01,2,MIA,21,None,61,4,737,1.3,0.085,-4.1,-0.4
1,achiupr01,3,TOR,22,None,73,28,1725,2.5,0.070,-2.6,-0.2
2,achiupr01,4,TOR,23,None,55,12,1140,2.2,0.093,-2.3,-0.1
3,achiupr01,5,2TM,24,None,74,18,1624,3.4,0.102,-1.4,0.2
4,adamsja01,2,MIL,24,None,7,0,18,-0.1,-0.252,-19.8,-0.1


In [82]:
print('shape:', player_season_stats_df.shape)
print('---')
print(player_season_stats_df.isna().sum())
print('---')
print('Duplicated:', player_season_stats_df.duplicated().sum())

shape: (2785, 12)
---
player_id           0
season_id           0
team_code           0
age                 0
position_code    2785
games               0
games_started       0
minutes             0
win_shares          0
ws_per_48           0
bpm                 0
vorp                0
dtype: int64
---
Duplicated: 0


### positions_df

In [18]:
positions_df

,position_id,position_code
0,1,C
1,2,F
2,3,G
3,4,PF
4,5,PG
5,6,SF
6,7,SG


In [79]:
print('shape:', positions_df.shape)
print('---')
print(positions_df.isna().sum())
print('---')
print('Duplicated:', positions_df.duplicated().sum())

shape: (7, 2)
---
position_id      0
position_code    0
dtype: int64
---
Duplicated: 0


### season_df

In [16]:
season_df.head()

,season_id,season_name
0,1,2019-20
1,2,2020-21
2,3,2021-22
3,4,2022-23
4,5,2023-24


In [80]:
print('shape:', season_df.shape)
print('---')
print(season_df.isna().sum())
print('---')
print('Duplicated:', season_df.duplicated().sum())

shape: (5, 2)
---
season_id      0
season_name    0
dtype: int64
---
Duplicated: 0


### team_df

In [17]:
team_df.head()


,team_id,team_code,team_name
0,1,2TM,None
1,2,3TM,None
2,3,4TM,None
3,4,ATL,None
4,5,BOS,None


In [81]:
print('shape:', team_df.shape)
print('---')
print(team_df.isna().sum())
print('---')
print('Duplicated:', team_df.duplicated().sum())

shape: (33, 3)
---
team_id       0
team_code     0
team_name    33
dtype: int64
---
Duplicated: 0


In [ ]:
# Tables with missing values

# champion_team_player_df -> a lot
# player_df -> birth_date, birth_place, weight_kg
# mvp_award_df -> mvp_rank
# player_season_stats -> position_code

# Inspecting Tables


In [ ]:
def numerical_data_summary(df: pd.DataFrame) -> pd.DataFrame:
    numerical_cols = df.select_dtypes(exclude=['str', 'object', 'bool'])

    summary_df = pd.DataFrame()

    for col in numerical_cols:
        summary_df[col] = {
            'Count': len(df[col]),
            'Unique': df[col].nunique(),
            'Min': df[col].min(),
            'Max': df[col].max(),
            'Mean': df[col].mean(),
            'Mode': df[col].mode()[0],
            'Q1': np.quantile(df[col].dropna(), 0.25),
            'Median': np.quantile(df[col].dropna(), 0.5),
            'Q3': np.quantile(df[col].dropna(), 0.75),
            'Variance': df[col].var(),
            'Std': df[col].std(),
            'range': df[col].max() - df[col].min(),
            'IQR': np.quantile(df[col].dropna(), 0.75) - np.quantile(df[col].dropna(), 0.25),
            'Skewness': df[col].skew(),
            'Kurtosis': df[col].kurtosis(),
        }

    return summary_df


In [85]:
print(numerical_data_summary(player_df))
print('---')
print(numerical_data_summary(player_season_stats_df))

            height_cm    weight_kg   country_id
Count     5411.000000  5411.000000  5411.000000
Unique      28.000000    81.000000    93.000000
Min        160.000000    51.000000     1.000000
Max        231.000000   163.000000    93.000000
Mean       198.138607    94.556089    80.372020
Mode       201.000000    83.000000    86.000000
Q1         190.000000    86.000000    86.000000
Median     198.000000    95.000000    86.000000
Q3         206.000000   102.000000    86.000000
Variance    83.632541   137.970375   317.641428
Std          9.145083    11.746079    17.822498
range       71.000000   112.000000    92.000000
IQR         16.000000    16.000000     0.000000
Skewness    -0.096902     0.481084    -3.192757
Kurtosis    -0.242055     0.365051     8.911063
---
            season_id          age        games  games_started        minutes  \
Count     2785.000000  2785.000000  2785.000000    2785.000000    2785.000000   
Unique       5.000000    25.000000    84.000000      84.000000    

In [86]:
missing_champ = champion_team_player_df[champion_team_player_df['games'].isna()]
print('Rows with missing stats:', len(missing_champ))
print(missing_champ.groupby(['season_id', 'team_code']).size())
print()

# Cross-check: do these players+seasons exist at all in player_season_stats_df?
merged_check = missing_champ.merge(
    player_season_stats_df[['player_id', 'season_id']],
    on=['player_id', 'season_id'], how='left', indicator=True
)
print(merged_check['_merge'].value_counts())
# if almost all say 'left_only', it confirms: these players simply have no season-stats row
# (never played that season) -> supports the DNP hypothesis rather than a scraping bug

Rows with missing stats: 76
season_id  team_code
2          MIL          22
3          GSW          17
4          DEN          18
5          BOS          19
dtype: int64

_merge
both          76
left_only      0
right_only     0
Name: count, dtype: int64


In [87]:
nan_mvp_rows = mvp_award_df[mvp_award_df['mvp_rank'].isna()]
print(nan_mvp_rows.sort_values(['season_id', 'mvp_share'], ascending=[True, False]))
print()
# Compare against the non-missing ranks in the same seasons to see where the gap sits
for sid in nan_mvp_rows['season_id'].unique():
    print(f'--- season_id {sid} ---')
    print(mvp_award_df[mvp_award_df['season_id'] == sid].sort_values('mvp_share', ascending=False))


    player_id  season_id  mvp_rank  mvp_share
31  hardeja01          2       NaN      0.001
33  jamesle01          2       NaN      0.001
41  leonaka01          2       NaN      0.001
14  derozde01          3       NaN      0.001
20  duranke01          3       NaN      0.001
34  jamesle01          3       NaN      0.001
6   brunsja01          4       NaN      0.001
46  moranja01          4       NaN      0.001

--- season_id 4 ---
    player_id  season_id  mvp_rank  mvp_share
25  embiijo01          4       1.0      0.915
38  jokicni01          4       2.0      0.674
3   antetgi01          4       3.0      0.606
58  tatumja01          4       4.0      0.280
27  gilgesh01          4       5.0      0.046
44  mitchdo01          4       6.0      0.030
52  sabondo01          4       7.0      0.027
18  doncilu01          4       8.0      0.010
12  curryst01          4       9.0      0.005
9   butleji01          4      10.0      0.003
26    foxde01          4      11.0      0.002
6   brunsja01

In [88]:
player_season_stats_df = player_season_stats_df.drop(columns=['position_code'])
print(player_season_stats_df.columns.tolist())

['player_id', 'season_id', 'team_code', 'age', 'games', 'games_started', 'minutes', 'win_shares', 'ws_per_48', 'bpm', 'vorp']


# Outlier Handling

In [89]:
LOW_SAMPLE_GAMES = 10
LOW_SAMPLE_MINUTES = 250

player_season_stats_df['low_sample'] = (
    (player_season_stats_df['games'] < LOW_SAMPLE_GAMES) |
    (player_season_stats_df['minutes'] < LOW_SAMPLE_MINUTES)
)
print(player_season_stats_df['low_sample'].value_counts())

low_sample
False    2093
True      692
Name: count, dtype: int64


In [90]:
# IQR-based flag as a supplementary check (for documentation, not automatic removal)
def iqr_outlier_flags(df, col):
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return (df[col] < lower) | (df[col] > upper)

for col in ['win_shares', 'bpm', 'vorp']:
    flag = iqr_outlier_flags(player_season_stats_df, col)
    print(f'{col}: {flag.sum()} IQR-flagged rows (kept, not removed -- likely genuine elite/poor seasons)')


win_shares: 81 IQR-flagged rows (kept, not removed -- likely genuine elite/poor seasons)
bpm: 181 IQR-flagged rows (kept, not removed -- likely genuine elite/poor seasons)
vorp: 227 IQR-flagged rows (kept, not removed -- likely genuine elite/poor seasons)


# Feature Engineering

In [91]:
stats_sorted = player_season_stats_df.sort_values(['player_id', 'season_id']).copy()
stats_sorted['played'] = stats_sorted['games'] > 0

# cumulative count of seasons actually played, inclusive of current season
stats_sorted['experience'] = stats_sorted.groupby('player_id')['played'].cumsum()

player_season_stats_df = stats_sorted.drop(columns=['played'])
player_season_stats_df[['player_id', 'season_id', 'games', 'experience']].head(10)

,player_id,season_id,games,experience
0,achiupr01,2,61,1
1,achiupr01,3,73,2
2,achiupr01,4,55,3
3,achiupr01,5,74,4
4,adamsja01,2,7,1
5,adamsst01,1,63,1
6,adamsst01,2,58,2
7,adamsst01,3,76,3
8,adamsst01,4,42,4
9,adebaba01,1,72,1


In [92]:
player_df['height_m'] = player_df['height_cm'] / 100
player_df['bmi'] = player_df['weight_kg'] / (player_df['height_m'] ** 2)
player_df['agility'] = player_df['height_cm'] / player_df['weight_kg']

player_df[['player_id', 'full_name', 'height_cm', 'weight_kg', 'bmi', 'agility']].head()

,player_id,full_name,height_cm,weight_kg,bmi,agility
0,abdelal01,Alaa Abdelnaby,208,108.0,24.963018,1.925926
1,abdulka01,Kareem Abdul-Jabbar,218,102.0,21.462840,2.137255
2,abdulma01,Walt Hazzard,188,83.0,23.483477,2.265060
3,abdulma02,Mahmoud Abdul-Rauf,185,73.0,21.329438,2.534247
4,abdulta01,Tariq Abdul-Wahad,198,101.0,25.762677,1.960396


In [93]:
player_season_stats_df['innate_ability'] = (
    player_season_stats_df['experience'] / player_season_stats_df['age']
)
player_season_stats_df[['player_id', 'season_id', 'age', 'experience', 'innate_ability']].head()

,player_id,season_id,age,experience,innate_ability
0,achiupr01,2,21,1,0.047619
1,achiupr01,3,22,2,0.090909
2,achiupr01,4,23,3,0.130435
3,achiupr01,5,24,4,0.166667
4,adamsja01,2,24,1,0.041667


In [94]:
mvp_award_df['is_mvp_candidate'] = True  # presence in this table already means they got votes
champion_team_player_df['is_champion'] = True

# Binning

In [95]:
height_bins = [0, 190, 201, 213, 300]
height_labels = ['Guard-height (<190cm)', 'Wing-height (190-201cm)', 'Big-height (201-213cm)', 'Very tall (>213cm)']

player_df['height_bin'] = pd.cut(player_df['height_cm'], bins=height_bins, labels=height_labels)
print(player_df['height_bin'].value_counts())


height_bin
Wing-height (190-201cm)    2045
Big-height (201-213cm)     1823
Guard-height (<190cm)      1419
Very tall (>213cm)          124
Name: count, dtype: int64


In [96]:
player_season_stats_df['experience_qbin'] = pd.qcut(
    player_season_stats_df['experience'], q=4, labels=['Rookie-tier', 'Developing', 'Prime', 'Veteran'], duplicates='drop'
)
print(player_season_stats_df['experience_qbin'].value_counts())

ValueError: Bin labels must be one fewer than the number of bin edges

# Master Table

In [97]:
master_df = (
    player_season_stats_df
    .merge(player_df.drop(columns=['height_m']), on='player_id', how='left')
    .merge(player_position_df, on='player_id', how='left')
    .merge(positions_df, on='position_id', how='left')
    .merge(
        mvp_award_df[['player_id', 'season_id', 'mvp_rank', 'mvp_share', 'is_mvp_candidate']],
        on=['player_id', 'season_id'], how='left'
    )
    .merge(
        champion_team_player_df[['player_id', 'season_id', 'team_code', 'is_champion']],
        on=['player_id', 'season_id', 'team_code'], how='left'
    )
    .merge(season_df, on='season_id', how='left')
)

# fill the join-created NaNs for the boolean flags with False (they mean "not a candidate / not a champion that season")
master_df['is_mvp_candidate'] = master_df['is_mvp_candidate'].fillna(False)
master_df['is_champion'] = master_df['is_champion'].fillna(False)

print(master_df.shape)
master_df.head()


(4266, 32)


,player_id,season_id,team_code,age,games,games_started,minutes,win_shares,ws_per_48,bpm,...,bmi,agility,height_bin,position_id,position_code,mvp_rank,mvp_share,is_mvp_candidate,is_champion,season_name
0,achiupr01,2,MIA,21,61,4,737,1.3,0.085,-4.1,...,26.693198,1.845455,Big-height (201-213cm),1,C,NaN,NaN,False,False,2020-21
1,achiupr01,2,MIA,21,61,4,737,1.3,0.085,-4.1,...,26.693198,1.845455,Big-height (201-213cm),4,PF,NaN,NaN,False,False,2020-21
2,achiupr01,3,TOR,22,73,28,1725,2.5,0.070,-2.6,...,26.693198,1.845455,Big-height (201-213cm),1,C,NaN,NaN,False,False,2021-22
3,achiupr01,3,TOR,22,73,28,1725,2.5,0.070,-2.6,...,26.693198,1.845455,Big-height (201-213cm),4,PF,NaN,NaN,False,False,2021-22
4,achiupr01,4,TOR,23,55,12,1140,2.2,0.093,-2.3,...,26.693198,1.845455,Big-height (201-213cm),1,C,NaN,NaN,False,False,2022-23


In [98]:
# Row count should not blow up vs the base table (player_season_stats_df), aside from
# legitimate one-to-many cases (a player with 2 positions -> 2 rows). Check and understand any growth.
print('player_season_stats_df rows:', len(player_season_stats_df))
print('master_df rows:', len(master_df))

# Spot-check: no player-season should be missing height/weight (player_df had no nulls there)
print('missing height in master:', master_df['height_cm'].isna().sum())
print('missing weight in master:', master_df['weight_kg'].isna().sum())

# Spot-check: mvp candidates and champions counts look sane
print('mvp candidate rows:', master_df['is_mvp_candidate'].sum())
print('champion rows:', master_df['is_champion'].sum())

player_season_stats_df rows: 2785
master_df rows: 4266
missing height in master: 0
missing weight in master: 0
mvp candidate rows: 120
champion rows: 99


In [99]:
# master_df.to_parquet('../Data/master_player_season.parquet', index=False)
master_df.to_csv('../Data/clean_data/master_player_season.csv', index=False)
print('Saved master_player_season.parquet and .csv')

Saved master_player_season.parquet and .csv
